In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


In [ ]:
project_root = Path.cwd().parent

print("PROJECT ROOT:", project_root)

input_file = project_root / "mapped_data" / "MAPPED_CHURN_DATA.xlsx"
output_file = project_root/'predicted_data'/'ML_Predicted_data.xlsx'
model_folder = project_root / "Models"/"Model_Datas"


PROJECT ROOT: c:\Users\VASU PATEL\OneDrive\Desktop\VS code\Customer-Churn-Prediction


In [3]:
data=pd.read_excel(input_file)

In [4]:
data.columns

Index(['CUSTOMER_ID', 'FIRST_NAME', 'LAST_NAME', 'GENDER', 'AGE',
       'MARITAL_STATUS', 'OCCUPATION', 'DEPENDENTS', 'MONTHLY_INCOME($)',
       'INCOME_CATEGORY', 'CITY', 'STATE', 'ZIP', 'JOIN_DATE', 'TENURE_MONTHS',
       'PHONE_SERVICE', 'MULTIPLE_LINES', 'INTERNET_SERVICE',
       'ONLINE_SECURITY', 'ONLINE_BACKUP', 'DEVICE_PROTECTION', 'TECH_SUPPORT',
       'TV_STREAMING', 'MOVIE_STREAMING', 'CONTRACT', 'PAPERLESS_BILLING',
       'PAYMENT_METHOD', 'NO_OF_SERVICES', 'MONTHLY_CHARGES($)',
       'TOTAL_CHARGES($)', 'CHURN_PROBABILITY', 'CHURN'],
      dtype='str')

In [5]:
data.shape

(30000, 32)

In [6]:
features=['GENDER', 'AGE','MARITAL_STATUS', 'OCCUPATION', 'DEPENDENTS', 'MONTHLY_INCOME($)',
            'INCOME_CATEGORY','TENURE_MONTHS','PHONE_SERVICE', 'MULTIPLE_LINES', 'INTERNET_SERVICE',
            'ONLINE_SECURITY', 'ONLINE_BACKUP', 'DEVICE_PROTECTION', 'TECH_SUPPORT','TV_STREAMING', 
            'MOVIE_STREAMING', 'CONTRACT', 'PAPERLESS_BILLING','PAYMENT_METHOD', 'NO_OF_SERVICES',
            'MONTHLY_CHARGES($)','TOTAL_CHARGES($)',]

target='CHURN'  

categorical_features = ['GENDER','MARITAL_STATUS', 'OCCUPATION',
            'INCOME_CATEGORY','PHONE_SERVICE', 'MULTIPLE_LINES', 'INTERNET_SERVICE',
            'ONLINE_SECURITY', 'ONLINE_BACKUP', 'DEVICE_PROTECTION', 'TECH_SUPPORT','TV_STREAMING', 
            'MOVIE_STREAMING', 'CONTRACT', 'PAPERLESS_BILLING','PAYMENT_METHOD']



In [7]:
data['CHURN_PROBABILITY']=data['CHURN_PROBABILITY'].round(2)
data['CHURN'] = data['CHURN'].map({'YES': 1, 'NO': 0})

#MODEL_1

In [8]:
x=data[features]
y=data[target]


In [9]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42,stratify=y)


In [10]:
model_cat = CatBoostClassifier(
    iterations=600,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_strength=1,
    auto_class_weights='Balanced',
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    verbose=100,
    early_stopping_rounds=100,
)

model_cat.fit(
    x_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(x_test, y_test)
)
joblib.dump(model_cat, model_folder/"churn_model.pkl")

0:	test: 0.7830831	best: 0.7830831 (0)	total: 183ms	remaining: 1m 49s
100:	test: 0.8015630	best: 0.8015630 (100)	total: 5.13s	remaining: 25.4s
200:	test: 0.8030591	best: 0.8030591 (200)	total: 11.3s	remaining: 22.5s
300:	test: 0.8032297	best: 0.8032958 (285)	total: 17.2s	remaining: 17.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8032957931
bestIteration = 285

Shrink model to first 286 iterations.


['c:\\Users\\VASU PATEL\\OneDrive\\Desktop\\VS code\\Customer-Churn-Prediction\\Models\\churn_model.pkl']

In [11]:
models = {
    "CatBoost": model_cat
}

for name, model in models.items():

    y_pred = model.predict(x_test)
    y_prob = model.predict_proba(x_test)[:, 1]

    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1-Score :", f1_score(y_test, y_pred))
    print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()

Accuracy : 0.702
Precision: 0.38392124692370794
Recall   : 0.7659574468085106
F1-Score : 0.5114754098360655
ROC-AUC  : 0.8032957931161577

Confusion Matrix:
[[3276 1502]
 [ 286  936]]



In [12]:
y_prob = model_cat.predict_proba(x_test)[:, 1]
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    
    y_pred = (y_prob >= threshold).astype(int)

    print(
        f"Threshold: {threshold} | "
        f"Accuracy: {accuracy_score(y_test, y_pred):.3f} | "
        f"Precision: {precision_score(y_test, y_pred):.3f} | "
        f"Recall: {recall_score(y_test, y_pred):.3f} | "
        f"F1: {f1_score(y_test, y_pred):.3f}"
    )

Threshold: 0.3 | Accuracy: 0.559 | Precision: 0.306 | Recall: 0.915 | F1: 0.458
Threshold: 0.4 | Accuracy: 0.638 | Precision: 0.343 | Recall: 0.849 | F1: 0.488
Threshold: 0.5 | Accuracy: 0.702 | Precision: 0.384 | Recall: 0.766 | F1: 0.511
Threshold: 0.6 | Accuracy: 0.764 | Precision: 0.446 | Recall: 0.652 | F1: 0.530
Threshold: 0.7 | Accuracy: 0.801 | Precision: 0.511 | Recall: 0.494 | F1: 0.502
Threshold: 0.8 | Accuracy: 0.820 | Precision: 0.635 | Recall: 0.267 | F1: 0.376


In [13]:
from sklearn.metrics import precision_recall_curve
import numpy as np

y_prob_test = model_cat.predict_proba(x_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, y_prob_test)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = np.argmax(f1)
best_threshold = thr[best_idx]
print("Best threshold:", best_threshold, "F1:", f1[best_idx])

Best threshold: 0.6139909221032558 F1: 0.5353570191742691


In [14]:
FINAL_THRESHOLD = best_threshold

y_prob_all = model_cat.predict_proba(data[features])[:, 1]

data['CHURN_PREDICTION'] = (
    y_prob_all >= FINAL_THRESHOLD
).astype(int)

#MODEL_2

In [15]:
x1=data[features]
y1=data['CHURN_PROBABILITY']

In [16]:
x1_train,x1_test,y1_train,y1_test=train_test_split(x1,y1,test_size=0.2,random_state=42)

In [17]:
model_prob=CatBoostRegressor(
    iterations=4000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=5,
    random_strength=1,
    loss_function='RMSE',
    random_state=42,
    verbose=100,
    early_stopping_rounds=1000,
)
model_prob.fit(
    x1_train,
    y1_train,
    cat_features=categorical_features,
    eval_set=(x1_test, y1_test)
)
joblib.dump(model_prob,model_folder/'Model_Prob.pkl')

0:	learn: 0.1698482	test: 0.1695584	best: 0.1695584 (0)	total: 50ms	remaining: 3m 19s
100:	learn: 0.0152927	test: 0.0142371	best: 0.0142371 (100)	total: 4.87s	remaining: 3m 7s
200:	learn: 0.0120017	test: 0.0111681	best: 0.0111681 (199)	total: 10.3s	remaining: 3m 13s
300:	learn: 0.0104203	test: 0.0097780	best: 0.0097780 (300)	total: 15.8s	remaining: 3m 13s
400:	learn: 0.0091561	test: 0.0086319	best: 0.0086319 (400)	total: 21.4s	remaining: 3m 11s
500:	learn: 0.0083284	test: 0.0079199	best: 0.0079199 (500)	total: 27s	remaining: 3m 8s
600:	learn: 0.0075849	test: 0.0073685	best: 0.0073684 (599)	total: 32.7s	remaining: 3m 5s
700:	learn: 0.0069446	test: 0.0068408	best: 0.0068407 (698)	total: 38.5s	remaining: 3m 1s
800:	learn: 0.0064831	test: 0.0065583	best: 0.0065583 (800)	total: 44.8s	remaining: 2m 58s
900:	learn: 0.0060324	test: 0.0062092	best: 0.0062091 (899)	total: 51.3s	remaining: 2m 56s
1000:	learn: 0.0057451	test: 0.0059942	best: 0.0059942 (1000)	total: 57.3s	remaining: 2m 51s
1100:	le

['c:\\Users\\VASU PATEL\\OneDrive\\Desktop\\VS code\\Customer-Churn-Prediction\\Models\\Model_Prob.pkl']

In [18]:
from sklearn.metrics import precision_recall_curve
import numpy as np

y1_prob_test = model_prob.predict(x1_test)

y1_actual_test = data.loc[y1_test.index, 'CHURN']
if y1_actual_test.dtype == object:
    y1_actual_test = y1_actual_test.map({'YES': 1, 'NO': 0})
y1_actual_test = y1_actual_test.astype(int)

prec2, rec2, thr2 = precision_recall_curve(y1_actual_test, y1_prob_test)
f1_2 = 2 * prec2 * rec2 / (prec2 + rec2 + 1e-9)
best_idx2 = np.argmax(f1_2)
best_threshold_model2 = thr2[best_idx2]

print("Model 2 best threshold:", best_threshold_model2, "F1:", f1_2[best_idx2])

Model 2 best threshold: 0.2691156902891031 F1: 0.5382220680045264


In [19]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# Model 2 predicted probabilities on the test set
y1_prob = model_prob.predict(x1_test)

# True churn labels for those same rows
y1_actual = data.loc[y1_test.index, 'CHURN'].astype(int)

THRESHOLD_MODEL2 = best_threshold_model2
y1_pred_class = (y1_prob >= THRESHOLD_MODEL2).astype(int)

acc  = accuracy_score(y1_actual, y1_pred_class)
prec = precision_score(y1_actual, y1_pred_class, zero_division=0)
rec  = recall_score(y1_actual, y1_pred_class, zero_division=0)
f1   = f1_score(y1_actual, y1_pred_class, zero_division=0)
roc  = roc_auc_score(y1_actual, y1_prob)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc:.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y1_actual, y1_pred_class))

Accuracy : 0.7775
Precision: 0.4563
Recall   : 0.6560
F1-Score : 0.5382
ROC-AUC  : 0.8100

Confusion Matrix:
 [[3887  927]
 [ 408  778]]


In [20]:
y1_pred = model_prob.predict(x1)
data["PREDICTED_CHURN_PROBABILITY"] = y1_pred

In [21]:
data['PREDICTED_CHURN_PROBABILITY']=data['PREDICTED_CHURN_PROBABILITY'].round(2)

In [22]:
data['PREDICTED_CHURN_PROBABILITY'].value_counts()

PREDICTED_CHURN_PROBABILITY
0.02    4846
0.03    1415
0.08    1271
0.04    1072
0.05    1012
        ... 
0.85       7
0.86       4
0.87       4
0.91       2
0.88       2
Name: count, Length: 89, dtype: int64

In [23]:

data["CHURN_PREDICTION"] = np.where(y_prob_all >= FINAL_THRESHOLD, "YES", "NO")

In [24]:
data['CHURN']=data['CHURN'].map({
    1:'YES',
    0:'NO',
})

In [25]:

conditions=[
    data['PREDICTED_CHURN_PROBABILITY']>=0.85,
    data['PREDICTED_CHURN_PROBABILITY']>=0.65,
    data['PREDICTED_CHURN_PROBABILITY']>=0.40,
    data['PREDICTED_CHURN_PROBABILITY']>=0.20,
    data['PREDICTED_CHURN_PROBABILITY']<0.20,
]

choices=['Very High','High','Medium','Low','Very Low']

data['RISK_TIER']=np.select(conditions,choices,default='')

In [26]:
data['RISK_TIER'].value_counts()

RISK_TIER
Very Low     17942
Low           7085
Medium        4028
High           926
Very High       19
Name: count, dtype: int64

In [27]:
data.to_excel(output_file, index=False)